In [1]:
ls

activator_20260427_164546.log   kourier4.txt
activator_20260427_165413.log   kourier5.txt
activator_20260427_170348.log   kourier6.txt
activator_20260427_171328.log   kourier7.txt
activator_20260427_172009.log   kourier8.txt
activator_20260427_172807.log   kourier9.txt
activator_20260427_173654.log   kourier.csv
activator_20260427_174351.log   latency_percentiles10.csv
activator_20260427_175224.log   latency_percentiles1.csv
activator_20260427_180128.log   latency_percentiles2.csv
autoscaler_20260427_164546.log  latency_percentiles3.csv
autoscaler_20260427_165413.log  latency_percentiles4.csv
autoscaler_20260427_170348.log  latency_percentiles5.csv
autoscaler_20260427_171328.log  latency_percentiles6.csv
autoscaler_20260427_172009.log  latency_percentiles7.csv
autoscaler_20260427_172807.log  latency_percentiles8.csv
autoscaler_20260427_173654.log  latency_percentiles9.csv
autoscaler_20260427_174351.log  logs/
autoscaler_20260427_175224.log  scheduler_20260427_164546.log
autoscaler_2026

In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

In [3]:
files = [
    "invocation_results_fixed1.csv",
    "invocation_results_fixed2.csv",
    "invocation_results_fixed3.csv",
    "invocation_results_fixed4.csv",
    "invocation_results_fixed5.csv",
    "invocation_results_fixed6.csv",
    "invocation_results_fixed7.csv",
    "invocation_results_fixed8.csv",
    "invocation_results_fixed9.csv",
    "invocation_results_fixed10.csv"
]
kourier = [
   "kourier.csv"
]

pods = [
    "logs/containers_live_20260427_164533.csv",
    "logs/containers_live_20260427_165410.csv",
    "logs/containers_live_20260427_170344.csv",
    "logs/containers_live_20260427_171318.csv",
    "logs/containers_live_20260427_172007.csv",
    "logs/containers_live_20260427_172804.csv",
    "logs/containers_live_20260427_173647.csv",
    "logs/containers_live_20260427_174346.csv",
    "logs/containers_live_20260427_175219.csv",
    "logs/containers_live_20260427_180125.csv"
  
   
]



scheduler = pd.read_csv('scheduler_times.csv')

# concaténation

#invocation
dfs = [pd.read_csv(f) for f in files]
all_data = pd.concat(dfs, ignore_index=True)

#kourier
kourier_dfs = [pd.read_csv(f) for f in kourier]
kourier_df = pd.concat(kourier_dfs, ignore_index=True)

#pod queue-proxy
dfs = [pd.read_csv(f) for f in pods]  
pod_df = pd.concat(dfs, ignore_index=True)  

#autoscaler
#dfs = [pd.read_csv(f) for f in auto]  
#auto_df = pd.concat(dfs, ignore_index=True)  

#activator
#dfs = [pd.read_csv(f) for f in acti]  
#acti_df = pd.concat(dfs, ignore_index=True)  

#acti = pd.read_csv('context_handler_times1.csv')
#auto = pd.read_csv('reconcile_times1.csv')
#df_percentil = pd.read_csv('latency_percentiles.csv')


In [4]:
#conversion en ms

all_data['execution_time_sec'] = all_data['execution_time_sec'] * 1000


pod_df["createdAt"] = pd.to_datetime(pod_df["createdAt"], errors="coerce", utc=True)
pod_df["startedAt"] = pd.to_datetime(pod_df["startedAt"], errors="coerce", utc=True)
pod_df = pod_df.dropna(subset=["createdAt", "startedAt"])

# Vérification
print(pod_df.dtypes[["createdAt", "startedAt"]])

pod_df["duration_ms"] = (pod_df["startedAt"] - pod_df["createdAt"]).dt.total_seconds() * 1000
# Vérifier le résultat
print(pod_df[["pod", "container_name", "duration_ms"]].head())
pod_df.to_csv("pod_df_100ms_apiserver_pinned.csv", index=False)

createdAt    datetime64[ns, UTC]
startedAt    datetime64[ns, UTC]
dtype: object
                                           pod container_name  duration_ms
0  grayscale-00001-deployment-6c9cd67ccc-8vjzk    queue-proxy   103.001654
1   pagerank-00001-deployment-54689999dc-799k6    queue-proxy   172.843499
2   graphgen-00001-deployment-545ff9c679-z55t8    queue-proxy   223.009244
3   genmatrix-00001-deployment-7db687b66-xggdz    queue-proxy    95.495923
4     resnet-00001-deployment-7fcbc8c8cb-j2wfn    queue-proxy    95.342361


In [5]:
all_data.to_csv("all_data_100ms_apiserver_pinned.csv", index=False)
#acti_df.to_csv("acti_df_10ms_1core.csv", index=False)
#auto_df.to_csv("auto_df_10ms_1core.csv", index=False)
kourier_df.to_csv("kourier_df_100ms_apiserver_pinned.csv", index=False)